In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard Components
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output

# Visualization
import plotly.express as px

# Utility Modules
import base64
import pandas as pd

# Database Module
from CRUD_Python_Module import AnimalShelter

##############################
# Database Configuration
##############################

USERNAME = "aacuser"
PASSWORD = "SimplePassword1"

db = AnimalShelter(USERNAME, PASSWORD)

##############################
# Rescue Breed Definitions
##############################

RESCUE_BREEDS = {

    "water": [
        "Labrador Retriever",
        "Chesapeake Bay Retriever",
        "Newfoundland"
    ],

    "mountain": [
        "German Shepherd",
        "Border Collie",
        "Australian Shepherd"
    ],

    "disaster": [
        "Bloodhound",
        "Beagle"
    ]
}

##############################
# Rescue Scoring Algorithm
##############################

def calculate_rescue_score(row, rescue_type):

    score = 0

    breed = str(row.get("breed", ""))
    age = row.get("age_upon_outcome_in_weeks", 0)
    outcome = str(row.get("outcome_type", ""))
    name = row.get("name", "")

    rescue_breeds = RESCUE_BREEDS.get(
        rescue_type,
        []
    )

    for rescue_breed in rescue_breeds:

        if rescue_breed.lower() in breed.lower():
            score += 50
            break

    if age <= 156:
        score += 30

    elif age <= 364:
        score += 15

    else:
        score += 5

    if outcome == "Adoption":
        score += 10

    elif outcome == "Transfer":
        score += 5

    if pd.notna(name) and str(name).strip():
        score += 10

    return score

##############################
# Ranking Algorithm
##############################

def rank_animals(df, rescue_type):

    if rescue_type == "reset":
        return df

    df = df.copy()

    df["rescue_score"] = df.apply(
        lambda row:
        calculate_rescue_score(
            row,
            rescue_type
        ),
        axis=1
    )

    df = df.sort_values(
        by="rescue_score",
        ascending=False
    )

    return df

##############################
# Data Retrieval
##############################

def get_filtered_data(filter_type="reset"):

    try:

        records = db.read(
            {},
            {"_id": 0}
        )

        df = pd.DataFrame.from_records(
            records
        )

        if filter_type != "reset":

            breeds = RESCUE_BREEDS.get(
                filter_type,
                []
            )

            df = df[
                df["breed"].apply(
                    lambda breed:
                    any(
                        rescue_breed.lower()
                        in str(breed).lower()
                        for rescue_breed
                        in breeds
                    )
                )
            ]

            df = rank_animals(
                df,
                filter_type
            )

        return df

    except Exception as e:

        print(
            f"Dashboard query error: {e}"
        )

        return pd.DataFrame()

##############################
# Initial Data Load
##############################

df = get_filtered_data()

##############################
# Dashboard Layout
##############################

app = JupyterDash(__name__)

image_filename = (
    "Grazioso Salvare Logo.png"
)

encoded_image = base64.b64encode(
    open(image_filename, "rb").read()
)

app.layout = html.Div([

    html.Center(
        html.B(
            html.H1(
                "CS-340 Dashboard"
            )
        )
    ),

    html.Hr(),

    html.Div([

        html.Img(
            src=
            f"data:image/png;base64,{encoded_image.decode()}",
            style={"height": "100px"}
        ),

        html.P(
            "Developer: Jake Whaley"
        )

    ],
    style={"textAlign": "center"}),

    html.Hr(),

    dcc.RadioItems(

        id="filter-type",

        options=[

            {
                "label": "Water Rescue",
                "value": "water"
            },

            {
                "label":
                "Mountain/Wilderness Rescue",
                "value": "mountain"
            },

            {
                "label":
                "Disaster/Individual Tracking",
                "value": "disaster"
            },

            {
                "label": "Reset",
                "value": "reset"
            }
        ],

        value="reset",

        labelStyle={
            "display": "inline-block",
            "margin-right": "10px"
        }
    ),

    html.Br(),

    dcc.Input(
        id="breed-search",
        type="text",
        placeholder="Search Breed...",
        debounce=True
    ),

    html.Br(),
    html.Br(),

    html.Hr(),

    dash_table.DataTable(

        id="datatable-id",

        columns=[
            {
                "name": i,
                "id": i
            }
            for i in df.columns
        ],

        data=df.to_dict("records"),

        page_size=10,

        sort_action="native",

        filter_action="native",

        row_selectable="single",

        selected_rows=[0],

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left"
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(

        style={
            "display": "flex"
        },

        children=[
            html.Div(
                id="graph-id"
            ),

            html.Div(
                id="map-id"
            )
        ]
    )
])

##############################
# Update Table
##############################

@app.callback(
    [
        Output(
            "datatable-id",
            "data"
        ),
        Output(
            "datatable-id",
            "columns"
        )
    ],
    [
        Input(
            "filter-type",
            "value"
        ),
        Input(
            "breed-search",
            "value"
        )
    ]
)
def update_dashboard(
    filter_type,
    breed_search
):

    if breed_search and breed_search.strip():

        records = db.search_by_breed(
            breed_search
        )

        df_filtered = pd.DataFrame(
            records
        )

        if "_id" in df_filtered.columns:
            df_filtered.drop(
                columns=["_id"],
                inplace=True
            )

    else:

        df_filtered = get_filtered_data(
            filter_type
        )

    columns = [

        {
            "name": i,
            "id": i
        }

        for i in df_filtered.columns
    ]

    return (
        df_filtered.to_dict("records"),
        columns
    )

##############################
# MongoDB Aggregation Chart
##############################

@app.callback(
    Output(
        "graph-id",
        "children"
    ),
    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        )
    ]
)
def update_graphs(viewData):

    top_breeds = db.get_top_breeds()

    breed_df = pd.DataFrame(
        top_breeds
    )

    if breed_df.empty:
        return []

    breed_df.rename(
        columns={
            "_id": "breed"
        },
        inplace=True
    )

    return [

        dcc.Graph(

            figure=px.pie(

                breed_df,

                names="breed",

                values="count",

                title=
                "Top 10 Breeds (MongoDB Aggregation)"
            )
        )
    ]

##############################
# Column Highlighting
##############################

@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional"
    ),
    [
        Input(
            "datatable-id",
            "selected_columns"
        )
    ]
)
def update_styles(
    selected_columns
):

    if not selected_columns:
        return []

    return [

        {
            "if": {
                "column_id": col
            },

            "background_color":
            "#D2F3FF"
        }

        for col
        in selected_columns
    ]

##############################
# Map Update
##############################

@app.callback(
    Output(
        "map-id",
        "children"
    ),
    [

        Input(
            "datatable-id",
            "derived_virtual_data"
        ),

        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(
    viewData,
    index
):

    if not viewData:
        return []

    dff = pd.DataFrame(
        viewData
    )

    if (
        "location_lat"
        not in dff.columns
        or
        "location_long"
        not in dff.columns
    ):
        return []

    row = (
        index[0]
        if index
        else 0
    )

    latitude = dff.iloc[row][
        "location_lat"
    ]

    longitude = dff.iloc[row][
        "location_long"
    ]

    if (
        pd.isna(latitude)
        or
        pd.isna(longitude)
    ):
        return []

    return [

        dl.Map(

            style={
                "width": "1000px",
                "height": "500px"
            },

            center=[
                latitude,
                longitude
            ],

            zoom=10,

            children=[

                dl.TileLayer(),

                dl.Marker(

                    position=[
                        latitude,
                        longitude
                    ],

                    children=[

                        dl.Tooltip(
                            str(
                                dff.iloc[row].get(
                                    "breed",
                                    "Unknown Breed"
                                )
                            )
                        ),

                        dl.Popup([

                            html.H1(
                                "Animal Information"
                            ),

                            html.P(
                                str(
                                    dff.iloc[row].get(
                                        "name",
                                        "Unknown"
                                    )
                                )
                            )
                        ])
                    ]
                )
            ]
        )
    ]

##############################
# Run Dashboard
##############################

from IPython.display import (
    display,
    HTML
)

codio_url = (
    "https://extendbread-tradesilver-8050.codio.io"
)

display(
    HTML(
        f'<a href="{codio_url}" target="_blank">'
        'Click here to open the Dashboard</a>'
    )
)

app.run_server(
    debug=True,
    host="0.0.0.0",
    port=8050,
    mode="external"
)